# Phase 3 — Monte Carlo & Path-Dependent Options

So far every price came from **Black-Scholes**, a closed-form formula. But Black-Scholes only
works for *vanilla* European options — ones whose payoff depends solely on the **final** stock
price. Many real options depend on the **entire price path**:

- **Asian options** pay off on the *average* price over the life of the option.
- **Barrier options** turn on or off depending on whether the price ever *touches* a level.

For these, there's often **no formula at all**. This is where **Monte Carlo** comes in — and
why having a simulation engine actually matters instead of being a toy reimplementation of a
formula you already have.

This notebook covers:

1. **The idea of Monte Carlo pricing** — simulate, average, discount
2. **Simulating GBM** — the same process Black-Scholes assumes
3. **Validating against Black-Scholes** — the convergence chart (proof the engine is correct)
4. **Error bars** — standard error and why a MC price without them is meaningless
5. **Variance reduction** — antithetic variates
6. **Path-dependent options** — Asian and barrier, which BS *cannot* price

> Prerequisite: notebooks 01 and 02. Everything here runs **offline** (no market data needed).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from optvol.pricing.black_scholes import bs_price
from optvol.pricing.monte_carlo import (
    simulate_paths, simulate_terminal,
    mc_european_price, mc_asian_price, mc_barrier_price,
)
print("Phase 3 — Monte Carlo")

## 1. The core idea

An option's fair value is the **discounted expected payoff** under the risk-neutral measure:

$$\text{price} = e^{-rT}\,\mathbb{E}[\text{payoff}]$$

Black-Scholes computes that expectation with calculus. **Monte Carlo computes it by brute
force** instead:

1. **Simulate** thousands of possible future price paths.
2. **Compute the payoff** on each path.
3. **Average** the payoffs (that estimates the expectation).
4. **Discount** back to today with $e^{-rT}$.

The beauty: step 2 can be *any* rule at all — average price, "did it touch 130?", anything —
so Monte Carlo prices contracts that have no formula. The cost: the answer is an *estimate*
with statistical noise, which shrinks as you add paths.

## 2. Simulating GBM

We assume the stock follows **geometric Brownian motion (GBM)** — the exact process
Black-Scholes is built on. Stepping through time with step $dt$:

$$S_{t+dt} = S_t \cdot \exp\!\Big((r - q - \tfrac{1}{2}\sigma^2)\,dt + \sigma\sqrt{dt}\,Z\Big),\quad Z\sim N(0,1)$$

`simulate_paths` does exactly this. Let's draw a handful of paths to see the "cloud" of
possible futures.

In [ ]:
S0, r, q, sigma, T = 100.0, 0.05, 0.0, 0.20, 1.0
paths = simulate_paths(S0, r, q, sigma, T, n_steps=252, n_paths=200, seed=0)

t = np.linspace(0, T, paths.shape[1])
plt.figure(figsize=(9, 5))
plt.plot(t, paths.T, lw=0.6, alpha=0.5)
plt.plot(t, paths.mean(axis=0), "k", lw=2.5, label="mean path")
plt.axhline(S0, ls=":", color="gray")
plt.xlabel("time (years)"); plt.ylabel("stock price")
plt.title("200 simulated GBM price paths")
plt.legend(); plt.grid(alpha=0.3); plt.show()

# Sanity: the mean should drift up at the risk-free rate (q=0 here).
print("mean final price:", round(paths[:, -1].mean(), 2),
      " vs theoretical S0*e^((r-q)T) =", round(S0 * np.exp((r - q) * T), 2))

## 3. Validating against Black-Scholes (the convergence proof)

Before trusting Monte Carlo on options that have *no* formula, we prove it agrees with the
formula on the one case that *does* — the vanilla European option. As we add paths, the MC
estimate should converge to the exact Black-Scholes value, and its error should shrink like
$1/\sqrt{n}$.

This chart is the single most important artifact of the phase: it's how you *demonstrate* the
engine is correct.

In [ ]:
bs = bs_price(S0, 100, T, r, sigma, "call", q)

path_counts = [500, 1000, 2000, 5000, 10000, 20000, 50000, 100000, 200000]
prices, errs = [], []
for n in path_counts:
    res = mc_european_price(S0, 100, T, r, sigma, "call", q, n_paths=n, seed=1)
    prices.append(res.price); errs.append(res.std_error)

plt.figure(figsize=(9, 5))
plt.errorbar(path_counts, prices, yerr=1.96*np.array(errs), fmt="o-",
             capsize=4, label="MC price ± 95% CI")
plt.axhline(bs, color="firebrick", ls="--", label=f"Black-Scholes = {bs:.4f}")
plt.xscale("log"); plt.xlabel("number of paths (log scale)"); plt.ylabel("call price")
plt.title("Monte Carlo converges to Black-Scholes as paths increase")
plt.legend(); plt.grid(alpha=0.3); plt.show()

print(f"Black-Scholes:  {bs:.4f}")
print(f"MC @ 200k:      {prices[-1]:.4f}  (95% CI half-width {1.96*errs[-1]:.4f})")

Notice two things: the dots close in on the red line, and the **error bars shrink**. Let's
confirm the error really follows the $1/\sqrt{n}$ law — on a log-log plot it should be a
straight line with slope $-1/2$.

In [ ]:
plt.figure(figsize=(8, 5))
plt.loglog(path_counts, errs, "o-", label="observed standard error")
# reference line proportional to 1/sqrt(n)
ref = errs[0] * np.sqrt(path_counts[0]) / np.sqrt(path_counts)
plt.loglog(path_counts, ref, "--", color="gray", label="∝ 1/√n reference")
plt.xlabel("number of paths"); plt.ylabel("standard error")
plt.title("Monte Carlo error decays like 1/√n")
plt.legend(); plt.grid(alpha=0.3, which="both"); plt.show()

## 4. Error bars are not optional

A Monte Carlo price is a **statistical estimate**, so quoting it without its uncertainty is
meaningless. The engine returns an `MCResult` carrying the price, its **standard error**, and a
**95% confidence interval**. Interpretation: run the whole simulation again with fresh randomness
and the answer will land in that interval ~95% of the time.

To halve the confidence interval you need **4×** the paths (because error $\propto 1/\sqrt{n}$) —
Monte Carlo buys accuracy slowly. That's the motivation for variance reduction, next.

In [ ]:
res = mc_european_price(S0, 100, T, r, sigma, "call", q, n_paths=100_000, seed=1)
print(res)                       # __repr__ shows price, se, CI, n
print("95% CI:", tuple(round(x, 4) for x in res.ci95))

## 5. Variance reduction: antithetic variates

A cheap trick to get a tighter estimate **for free**. For every random draw $Z$ we also use
$-Z$. If one path was pushed high by luck, its mirror is pushed low, so their errors partly
cancel — the average is more stable. Same number of paths, smaller error.

The engine does this by default (`antithetic=True`). Let's measure the improvement.

In [ ]:
n = 40_000
with_anti = np.array([mc_european_price(S0, 100, T, r, sigma, "call", q,
                      n_paths=n, antithetic=True,  seed=s).price for s in range(40)])
without   = np.array([mc_european_price(S0, 100, T, r, sigma, "call", q,
                      n_paths=n, antithetic=False, seed=s).price for s in range(40)])

print(f"Std dev of the estimate across 40 runs (lower = better):")
print(f"  plain Monte Carlo:  {without.std():.4f}")
print(f"  antithetic:         {with_anti.std():.4f}")
print(f"  variance reduction: {(1 - with_anti.var()/without.var())*100:.0f}% less variance")

## 6. The payoff: options with NO formula

Now the whole point. These payoffs depend on the *path*, so Black-Scholes can't touch them — but
Monte Carlo prices them with the same machinery, just a different payoff rule.

### Asian option — pays on the *average* price
$$\text{Asian call payoff} = \max\!\big(\overline{S} - K,\ 0\big),\quad \overline{S}=\text{average of }S\text{ along the path}$$

Averaging smooths out extremes, so an Asian option is **cheaper** than the vanilla equivalent
(and harder to manipulate near expiry — a reason they're used for commodities and FX).

In [ ]:
euro = bs_price(S0, 100, T, r, sigma, "call", q)
asian = mc_asian_price(S0, 100, T, r, sigma, "call", q, n_paths=200_000, n_steps=100, seed=2)
print(f"Vanilla European call (Black-Scholes): {euro:.4f}")
print(f"Asian call (Monte Carlo):              {asian}")
print(f"Asian is cheaper, as expected:         {asian.price < euro}")

### Barrier option — turns on/off if a level is touched

A **knock-out** call dies if the price ever hits the barrier; a **knock-in** only comes alive if
it does. A neat internal check: **knock-in + knock-out = vanilla** (every path either touches the
barrier or doesn't), which the test suite verifies exactly.

In [ ]:
barrier = 130.0
common = dict(S0=S0, K=100, T=T, r=r, sigma=sigma, option_type="call",
              n_paths=200_000, n_steps=150, seed=3)
ko = mc_barrier_price(barrier=barrier, barrier_type="up-and-out", **common)
ki = mc_barrier_price(barrier=barrier, barrier_type="up-and-in",  **common)

print(f"Up-and-OUT call (dies if S hits {barrier:.0f}): {ko.price:.4f}")
print(f"Up-and-IN  call (lives only if S hits {barrier:.0f}): {ki.price:.4f}")
print(f"knock-in + knock-out = {ko.price + ki.price:.4f}")
print(f"vanilla European     = {euro:.4f}   <- should match (in+out parity)")

Let's visualize *why* the knock-out is cheaper: some paths breach the barrier and forfeit their
payoff entirely.

In [ ]:
paths = simulate_paths(S0, r, q, sigma, T, n_steps=150, n_paths=60, seed=5)
t = np.linspace(0, T, paths.shape[1])
breached = np.any(paths >= barrier, axis=1)

plt.figure(figsize=(9, 5))
for p, b in zip(paths, breached):
    plt.plot(t, p, lw=0.8, color=("firebrick" if b else "steelblue"),
             alpha=0.7)
plt.axhline(barrier, color="black", ls="--", label=f"barrier = {barrier:.0f}")
plt.xlabel("time (years)"); plt.ylabel("stock price")
plt.title("Up-and-out: red paths breached the barrier and pay 0")
plt.legend(); plt.grid(alpha=0.3); plt.show()
print(f"{breached.mean()*100:.0f}% of these paths knocked out.")

## Recap — the one-paragraph version

> Black-Scholes only prices vanilla options whose payoff depends on the **final** price. For
> **path-dependent** options — Asian (average price) and barrier (touch a level) — there's often
> no formula, so we use **Monte Carlo**: simulate many GBM price paths, apply the payoff rule to
> each, average, and discount. We **validated** the engine by showing its price for a vanilla
> European option converges to Black-Scholes as paths increase, with error shrinking like
> $1/\sqrt{n}$, and we sharpen estimates with **antithetic variates**. Every price comes with a
> **standard error and confidence interval**, because a Monte Carlo number without error bars is
> meaningless.

### Skills this phase signals to a recruiter
- Stochastic simulation (GBM) and risk-neutral pricing by expectation
- Numerical **convergence** analysis and error quantification
- **Variance-reduction** techniques (antithetic variates)
- Pricing exotic, path-dependent derivatives that closed-form models can't handle

### Where to go next
- Open `optvol/pricing/monte_carlo.py` and match each pricer to Sections 3–6.
- Run `PYTEST_DISABLE_PLUGIN_AUTOLOAD=1 python -m pytest tests/test_monte_carlo.py -q`.
- Phase 4 (roadmap): fit the **SVI** vol surface and an **ML** surface model, then benchmark
  Black-Scholes vs. Monte Carlo vs. the fitted model on real data.